[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaylink-coder/miss-yaya/blob/main/yaya-ai/notebooks/train_colab.ipynb)

# Yaya-125M — Real Reasoning Training

**Three phases that make Yaya actually reason:**

| Phase | What | Time (T4) | Why |
|-------|------|-----------|-----|
| 1 | **Pretrain** on Wikipedia | ~3 hrs | World knowledge — facts, language, concepts |
| 2 | **Download** 260k reasoning examples | ~15 min | GSM8K + MetaMath + OpenHermes |
| 3 | **SFT** on reasoning data | ~4 hrs | Teaches CoT format + calc tool use |

Each session auto-resumes from Google Drive. Run across multiple sessions.

**Required:** Runtime → Change runtime type → **T4 GPU**

## Step 0 — Check GPU & setup

In [ ]:
import torch, os, sys, glob

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f'GPU : {name}  ({vram} GB VRAM)')
    print(f'PyTorch: {torch.__version__}')
    DTYPE = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
    IS_A100 = 'A100' in name
else:
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → T4 GPU')

print(f'dtype: {DTYPE}  |  A100: {IS_A100}')

## Step 1 — Mount Drive + clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All checkpoints survive session resets here
DRIVE_DIR     = '/content/drive/MyDrive/yaya-125m'
PRETRAIN_DIR  = f'{DRIVE_DIR}/pretrain'
SFT_DIR       = f'{DRIVE_DIR}/sft'
DATA_CACHE    = f'{DRIVE_DIR}/data_cache'

for d in [PRETRAIN_DIR, SFT_DIR, DATA_CACHE]:
    os.makedirs(d, exist_ok=True)

print('Drive dirs ready:')
print(f'  Pretrain:  {PRETRAIN_DIR}')
print(f'  SFT:       {SFT_DIR}')
print(f'  Data cache:{DATA_CACHE}')

In [ ]:
import os

if not os.path.exists('/content/miss-yaya'):
    !git clone https://github.com/jaylink-coder/miss-yaya.git /content/miss-yaya
else:
    !cd /content/miss-yaya && git pull

!pip install -q sentencepiece pyyaml datasets

os.chdir('/content/miss-yaya/yaya-ai')
sys.path.insert(0, '/content/miss-yaya/yaya-ai')
print('Ready:', os.getcwd())

## Phase 1 — Pretrain on Wikipedia
Gives Yaya world knowledge: facts, language structure, concepts.
Without this, SFT is teaching format to a blank brain.

**Skip this cell if `PRETRAIN_DIR` already has checkpoints** (auto-detected).

In [ ]:
import yaml, glob

# Check if pretrain already done or in progress
pretrain_ckpts = sorted(glob.glob(f'{PRETRAIN_DIR}/checkpoint-*'))

# Load and patch pretrain config
with open('configs/training/train_125m.yaml') as f:
    pt_cfg = yaml.safe_load(f)

pt_cfg['checkpointing']['save_dir'] = PRETRAIN_DIR
pt_cfg['training']['dtype']         = DTYPE
pt_cfg['training']['gradient_checkpointing'] = True

# T4: batch=4, accum=8 → eff=32 seqs × 1024 tok = 32k tok/step
# A100: batch=8, accum=4
pt_cfg['training']['per_device_batch_size']      = 8 if IS_A100 else 4
pt_cfg['training']['gradient_accumulation_steps'] = 4 if IS_A100 else 8
pt_cfg['training']['max_steps'] = 5000   # ~160M tokens — enough for world knowledge
pt_cfg['training']['max_seq_length'] = 1024

with open('configs/training/train_125m.yaml', 'w') as f:
    yaml.dump(pt_cfg, f)

if pretrain_ckpts:
    resume = f'--resume {pretrain_ckpts[-1]}'
    print(f'Resuming pretrain from step ~{pretrain_ckpts[-1].split("-")[-1]}')
else:
    resume = ''
    print('Starting pretrain from scratch — downloading Wikipedia...')

print(f'Config: batch={pt_cfg["training"]["per_device_batch_size"]}, '
      f'accum={pt_cfg["training"]["gradient_accumulation_steps"]}, '
      f'dtype={DTYPE}')

In [ ]:
# Download + tokenize Wikipedia (first run only — cached to Drive after)
import os
WIKI_TRAIN = f'{DATA_CACHE}/wiki_train'
WIKI_EVAL  = f'{DATA_CACHE}/wiki_eval'

if not os.path.exists(f'{WIKI_TRAIN}/shard_00000.bin'):
    print('Downloading + tokenizing Wikipedia 20231101.en (20% sample)...')
    from datasets import load_dataset
    from src.tokenizer.tokenizer import YayaTokenizer
    import numpy as np

    os.makedirs(WIKI_TRAIN, exist_ok=True)
    os.makedirs(WIKI_EVAL, exist_ok=True)

    ds = load_dataset('wikimedia/wikipedia', '20231101.en', split='train[:20%]')
    print(f'Loaded {len(ds):,} articles')

    tok = YayaTokenizer('data/tokenizer/yaya_tokenizer.model')
    split = ds.train_test_split(test_size=0.005, seed=42)

    def tokenize_and_save(dataset, out_dir, fname):
        tokens = []
        for i, row in enumerate(dataset):
            tokens.extend(tok.encode(row['text']))
            if (i + 1) % 10000 == 0:
                print(f'  {i+1:,} articles, {len(tokens)/1e6:.0f}M tokens')
        arr = np.array(tokens, dtype=np.uint16)
        arr.tofile(f'{out_dir}/{fname}')
        print(f'  Saved {len(arr)/1e6:.0f}M tokens → {out_dir}/{fname}')

    tokenize_and_save(split['train'], WIKI_TRAIN, 'shard_00000.bin')
    tokenize_and_save(split['test'],  WIKI_EVAL,  'eval.bin')
else:
    size_mb = os.path.getsize(f'{WIKI_TRAIN}/shard_00000.bin') / 1e6
    print(f'Wikipedia already cached ({size_mb:.0f} MB) — skipping download')

# Point config at cached data
with open('configs/training/train_125m.yaml') as f:
    pt_cfg = yaml.safe_load(f)
pt_cfg['data']['train_data'] = WIKI_TRAIN
pt_cfg['data']['eval_data']  = WIKI_EVAL
with open('configs/training/train_125m.yaml', 'w') as f:
    yaml.dump(pt_cfg, f)
print('Config updated with Wikipedia paths')

In [ ]:
# Run pretrain — streams progress live
!WANDB_DISABLED=true python -u scripts/train_sft.py \
    --model_config configs/model/yaya_125m.yaml \
    --train_config configs/training/train_125m.yaml \
    {resume}

## Phase 2 — Download 260k reasoning examples
GSM8K (8.5k) + MetaMath (200k) + OpenHermes (50k) + Yaya existing (3.5k)

All converted to Yaya format with `<|think|>` and `<|calc|>` tokens.

**Skip if `DATA_CACHE/yaya_reasoning_large.jsonl` already exists.**

In [ ]:
import os, shutil

LARGE_DATA = f'{DATA_CACHE}/yaya_reasoning_large.jsonl'
LOCAL_DATA = 'data/sft/yaya_reasoning_large.jsonl'

if os.path.exists(LARGE_DATA):
    # Already downloaded in a previous session — copy to local
    shutil.copy(LARGE_DATA, LOCAL_DATA)
    import subprocess
    n = int(subprocess.check_output(['wc', '-l', LOCAL_DATA]).split()[0])
    print(f'Data already cached: {n:,} examples → copied to {LOCAL_DATA}')
else:
    print('Downloading reasoning datasets (GSM8K + MetaMath + OpenHermes)...')
    !python scripts/download_reasoning_data.py
    # Cache to Drive so we don't re-download next session
    shutil.copy(LOCAL_DATA, LARGE_DATA)
    print(f'Cached to Drive: {LARGE_DATA}')

## Phase 3 — SFT on reasoning data
Fine-tunes the pretrained base on 260k reasoning + CoT + calc-tool examples.
This is what teaches Yaya **when** and **how** to reason.

Auto-resumes from latest SFT checkpoint on re-run.

In [ ]:
import yaml, glob

# Load SFT config
SFT_CFG = 'configs/training/sft_reasoning_large.yaml'
with open(SFT_CFG) as f:
    sft_cfg = yaml.safe_load(f)

sft_cfg['checkpointing']['save_dir'] = SFT_DIR
sft_cfg['training']['dtype']         = DTYPE
sft_cfg['data']['train_data']        = LOCAL_DATA
sft_cfg['data']['eval_data']         = LOCAL_DATA

# T4: batch=4, accum=8. A100: batch=8, accum=4
sft_cfg['training']['per_device_batch_size']      = 8 if IS_A100 else 4
sft_cfg['training']['gradient_accumulation_steps'] = 4 if IS_A100 else 8

with open(SFT_CFG, 'w') as f:
    yaml.dump(sft_cfg, f)

# Find start point: SFT checkpoint > pretrain checkpoint > scratch
sft_ckpts     = sorted(glob.glob(f'{SFT_DIR}/checkpoint-*'))
pretrain_ckpts= sorted(glob.glob(f'{PRETRAIN_DIR}/checkpoint-*'))

if sft_ckpts:
    mode = f'--resume {sft_ckpts[-1]}'
    print(f'Resuming SFT from: {sft_ckpts[-1]}')
elif pretrain_ckpts:
    mode = f'--pretrain_checkpoint {pretrain_ckpts[-1]}'
    print(f'Starting SFT from pretrain: {pretrain_ckpts[-1]}')
else:
    mode = ''
    print('WARNING: No pretrain checkpoint — SFT from random init (run Phase 1 first)')

eff = sft_cfg['training']['per_device_batch_size'] * sft_cfg['training']['gradient_accumulation_steps']
print(f'Effective batch: {eff} | dtype: {DTYPE} | max_steps: {sft_cfg["training"]["max_steps"]}')

In [ ]:
!WANDB_DISABLED=true python -u scripts/train_sft.py \
    --model_config configs/model/yaya_125m.yaml \
    --train_config configs/training/sft_reasoning_large.yaml \
    {mode}

## Test — Does Yaya actually reason?
Run after Phase 3. Tests compute, chain-of-thought, and logic.

In [ ]:
import torch, glob, os
from src.utils.config import load_model_config
from src.model.yaya_model import YayaForCausalLM
from src.tokenizer.tokenizer import YayaTokenizer, ASSISTANT_TOKEN
from src.inference.generator import TextGenerator, GenerationConfig
from src.inference.tool_generator import ToolAugmentedGenerator
from src.training.checkpointing import CheckpointManager

# Best checkpoint: SFT > pretrain
sft_ckpts     = sorted(glob.glob(f'{SFT_DIR}/checkpoint-*'))
pretrain_ckpts= sorted(glob.glob(f'{PRETRAIN_DIR}/checkpoint-*'))
ckpt_path     = (sft_ckpts or pretrain_ckpts or [None])[-1]

if not ckpt_path:
    print('No checkpoint found. Run training first.')
else:
    print(f'Loading: {ckpt_path}')
    device = 'cuda'
    model  = YayaForCausalLM(load_model_config('configs/model/yaya_125m.yaml')).to(device)
    CheckpointManager(save_dir=os.path.dirname(ckpt_path)).load(model, checkpoint_path=ckpt_path)
    model.eval()

    tok      = YayaTokenizer('data/tokenizer/yaya_tokenizer.model')
    tool_gen = ToolAugmentedGenerator(TextGenerator(model, tok, device=device), verbose=True)
    gen_cfg  = GenerationConfig(max_new_tokens=400, temperature=0.2,
                                top_p=0.9, repetition_penalty=1.3)

    SYSTEM = ('You are Yaya, a brilliant AI. Think step by step. '
              'Use <|calc|>EXPRESSION<|/calc|> for arithmetic.')

    TESTS = [
        # Compute
        ('What is 347 × 28?',                                          '9716'),
        ('What is 15% of 4,800?',                                      '720'),
        # Algebra
        ('Solve for x: 5x + 3 = 28',                                   '5'),
        # Word problem
        ('A shop buys goods for KSh 3,600 and sells for KSh 4,500. '
         'What is the profit percentage?',                             '25'),
        # Logic
        ('All squares are rectangles. Shape A is a square. '
         'Is Shape A a rectangle?',                                    'yes'),
        # Multi-step
        ('A farmer has 240 chickens. He sells 1/3, then buys 40 more. '
         'How many does he have?',                                     '200'),
    ]

    passed = 0
    for question, expected in TESTS:
        msgs   = [{'role':'system','content':SYSTEM},
                  {'role':'user',  'content':question}]
        prompt = tok.format_chat(msgs) + '\n' + ASSISTANT_TOKEN + '\n'
        ans    = tool_gen.generate(prompt, gen_cfg).strip()
        ok     = expected in ans
        passed += ok
        mark   = '✓' if ok else '✗'
        print(f'{mark} Q: {question[:60]}')
        print(f'  Expected: {expected}')
        print(f'  Got:      {ans[:200]}')
        print()

    print(f'Score: {passed}/{len(TESTS)} = {passed/len(TESTS):.0%}')

## Download checkpoint
Save the trained model to your local machine.

In [ ]:
import glob, shutil
from google.colab import files

ckpts = sorted(glob.glob(f'{SFT_DIR}/checkpoint-*'))
if not ckpts:
    ckpts = sorted(glob.glob(f'{PRETRAIN_DIR}/checkpoint-*'))

if ckpts:
    latest   = ckpts[-1]
    zip_path = '/content/yaya-125m-trained.zip'
    shutil.make_archive('/content/yaya-125m-trained', 'zip', latest)
    print(f'Zipping {latest} ...')
    files.download(zip_path)
else:
    print('No checkpoint to download.')